# Step 3: Immune Density Visualization
**Purpose:** Combine the per sample immune density CSVs from step 2 into a single table and make boxplots comparing immune cell infiltration across genotypes (final visual output of this pipeline)

## What this notebook does

1. **Configuration + setup:** Define input and output directories, the genotype display order, color palette, and which immune markers to plot
2. **Data aggregation:** Find every `*immune_densities*.csv` file in the data directory (one per sample from step 2), stack them into one combined dataframe (one row per ROI), and save a merged csv for reference
3. **Validation:** Confirm expected density columns and genotypes are actually present in the combined data, warn the user before any plots are generated if any missing
4. **Plot generation:** For each immune marker, build a layered figure containing: a colored boxplot per genotype, individual ROI points, and mean +- SD error bars 

## Inputs

* `<sample_id>_immune_densities.csv`: immune density tables from step 2 (one per sample)

## Outputs

* `combined_immune.csv`: single merged table with all ROIs from all samples
* `<MARKER>_density.png`: one boxplot per immune marker 

## Requirements

* Python 3.10 environment with: pandas, numpy, matplotlib
* No manual steps required for this part!

In [6]:
# Import dependencies
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Hide minor warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [14]:
# ‼️UPDATE‼️ this cell when doing a new sample 

# First set your working directory (this is the analysis directory we created in Step 1)
working_dir = Path('C:/Users/kelse/OneDrive - The Mount Sinai Hospital/02_Presentations/Platforma_MCMICRO_Tutorial_Docs/MCMICRO_Python_Analysis/analysis') # Please update root folder containing all the samples

DATA_DIR = Path(f'{working_dir}/immune') # folder containing per sample immune densities CSVs (step 2 output)

SAVE_DIR = Path(f'{working_dir}/plots') # output folder to save the plots we create

GENOTYPE_ORDER = ["test_run"] # genotype display order from left to right on x axis

GENOTYPE_COLORS = { # colors for each genotype (must match names above exactly)
    "test_run": "#d62728"
}

DENSITY_COLS = [ # the immune markers to plot -- these have to match column names in the combined CSV exactly
    "cd8_cells_per_mm2",
    "cd4_cells_per_mm2",
    "macrophage_cells_per_mm2",
    "nk_cells_per_mm2",
]

SAVE_DPI = 300 # dots per inch, 300 is the publication standard but 150 fine for internal review

# Confirm the configuration 
print(f"Data directory: {DATA_DIR}")
print(f"Save directory: {SAVE_DIR}")
print(f"Genotypes: {GENOTYPE_ORDER}")
print(f"Markers: {DENSITY_COLS}")

Data directory: C:\Users\kelse\OneDrive - The Mount Sinai Hospital\02_Presentations\Platforma_MCMICRO_Tutorial_Docs\MCMICRO_Python_Analysis\analysis\immune
Save directory: C:\Users\kelse\OneDrive - The Mount Sinai Hospital\02_Presentations\Platforma_MCMICRO_Tutorial_Docs\MCMICRO_Python_Analysis\analysis\plots
Genotypes: ['test_run']
Markers: ['cd8_cells_per_mm2', 'cd4_cells_per_mm2', 'macrophage_cells_per_mm2', 'nk_cells_per_mm2']


In [12]:
## Find all per sample immune density CSVs 
# Skip macOS AppleDouble files (they end in ._filename); these show up when the drive was accessed on a mac, but they're not actually CSVs!
    
csv_files = sorted(
    f for f in DATA_DIR.glob("*immune_densities*.csv") # looks for csv files that contain "immune_densities"
    if not f.name.startswith("._")                     # sorted alphabetically
    )  

print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(f" - {f.name}") # gives you filename without the full path

# This is like a checkpoint -- stop early with a clear message if no files could be found
if len(csv_files) == 0:
    raise FileNotFoundError( # trigger an exception -- stop running if there are no files
        f"No correct files found in:\n{DATA_DIR}\n"
        "Check that your DATA_DIR is correct and/or that step 2 ran correctly.") 

# Stack all of the CSVs into 1 dataframe 
df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True) # reset the row numbering so rows don't repeat within each file

print(f"\nCombined table has {df.shape[0]} rows and {df.shape[1]} columns")

# Save the combined CSV for reference and use later on
combined_path = DATA_DIR / "combined_immune.csv"
df.to_csv(combined_path, index=False)
print(f"Combined CSV has been saved to {combined_path}")

# Make sure that the expected density columns exist before we move on
missing_cols = [c for c in DENSITY_COLS if c not in df.columns]
if missing_cols:
    print("\nWarning, these expected columns are missing from the data:")
    for c in missing_cols:
        print(f" - {c}")
    print("Check step 2 output or update DENSITY_COLS in the configuration cell")
else:
    print("All expected density columns are present!")

# Check the genotypes that were found vs expected 
found_genotypes = df["Genotype"].unique().tolist() # get all the distinct values in genotype column and convert to a list
missing_genos = [g for g in GENOTYPE_ORDER if g not in found_genotypes]
if missing_genos:
    print(f"\nWarning, these genotypes are missing from the data: {missing_genos}")
else:
    print("All expected genotypes are present!")

# Preview the first few rows 
df.head()

Found 1 CSV file(s):
 - BDBAT0085_immune_densities.csv

Combined table has 4 rows and 16 columns
Combined CSV has been saved to C:\Users\kelse\OneDrive - The Mount Sinai Hospital\02_Presentations\Platforma_MCMICRO_Tutorial_Docs\MCMICRO_Python_Analysis\analysis\immune\combined_immune.csv
All expected density columns are present!
All expected genotypes are present!


,Genotype,ROI,Total_Cells,Area_mm2,cd8_count,cd8_fraction,cd8_cells_per_mm2,cd4_count,cd4_fraction,cd4_cells_per_mm2,macrophage_count,macrophage_fraction,macrophage_cells_per_mm2,nk_count,nk_fraction,nk_cells_per_mm2
0,test_run,ROI1,22994,2.090736,1524,0.066278,728.930012,350,0.015221,167.405186,7650,0.332695,3658.999076,33,0.001435,15.783918
1,test_run,ROI2,16012,1.226097,1748,0.109168,1425.662628,755,0.047152,615.775334,7798,0.487010,6360.021265,94,0.005871,76.666068
2,test_run,ROI3,16916,1.494806,1831,0.108241,1224.908107,301,0.017794,201.363922,7123,0.421081,4765.166820,41,0.002424,27.428308
3,test_run,ROI4,10394,1.026351,894,0.086011,871.047384,360,0.034635,350.757336,2520,0.242448,2455.301352,24,0.002309,23.383822


## Plot immune densities
**Purpose:** Loop over each marker in `DENSITY_COLS` and make 1 figure per marker, layering 3 views of the data on the same axes: boxplot (median + IQR), jittered scatter (every individual ROI), and mean +- SD error bars. Each figure is saved as a 300 dpi PNG image to `SAVE_DIR` file path

In [16]:
# Create the output directory if it doesn't exist 
os.makedirs(SAVE_DIR, exist_ok=True) # move on if already exists

# Set a random seed so jitter is identical across runs -- without this the scatter positions would change every run
# This would make each figure version look a little different even when the data didn't change
np.random.seed(42)

# Set this to True to use log y-axis, useful when densities span orders of magnitude across genotypes
USE_LOG_SCALE = False  # ‼️UPDATE‼️ THIS if needed

# Global font/style settings 
plt.rcParams.update({
    "font.size":       11,
    "axes.labelsize":  12,
    "axes.titlesize":  13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth":  1.2,
})

# ------------------------------------------------------------------------------------
# MAIN PLOTTING LOOP
# This iterates through each immune marker (CD8, CD4, etc) defined in the configuration
# and makes 1 boxplot figure per marker, saved as a PNG
# ------------------------------------------------------------------------------------

for col in DENSITY_COLS:  # iterate through each marker column, like "cd8_cells_per_mm2"

    # Step 1-- Safety check 
    if col not in df.columns: # Again, if column is missing from the dataframe, skip it and jump to the next marker
        print(f"SKIP {col} —- column not found in the data")
        continue                                        

    # Step 2-- Pull data for this marker, split by each genotype. End up with a list of arrays, one per genotype
    data = [ # select specific rows + columns; use boolean mask to pull all rows of a genotype, then marker column for those rows
        df.loc[df["Genotype"] == g, col].dropna().values for g in GENOTYPE_ORDER ]
                                                         # drop ROIs where marker wasn't measured
                                                         # convert pandas series to plain numpy array
                                                         # repeat for each genotype in display order
                                             
    

    # Step 3-- Print per marker sample counts 
    counts_str = ", ".join( # join strings together separated by commas
        f"{g}: n={len(d)}"  # format like "Genotype : number of ROIs it corresponds to"
        for g, d in zip(GENOTYPE_ORDER, data) # pair each genotype name with its array
    )
    print(f"{col} | {counts_str}") 

    # Step 4-- Set the x axis positions for each genotype group 
    x = np.arange(1, len(GENOTYPE_ORDER) + 1) # → generate a sequence of numbers for how many genotypes there are

    # Step 5-- Create the figure and axes 
    fig, ax = plt.subplots(figsize=(5.2, 4.5)) # fig is the canvas (for saving), and ax is the actual plot area in inches

    # Step 6-- Draw the boxplot 
    # NOTE: Normally the upper whisker extends from Q3 up to the highest data point that is still BELOW the outlier fence (Q3+1.5*IQR)
    # BUT: if the highest non-outlier is itself below Q3, the whisker
    #   collapses, and the upper cap gets drawn AT Q3 (right on top of the
    #   box edge) -- making it visually invisible.
    # - The same mirror rule applies to the lower whisker with Q1.
    box = ax.boxplot( # returns dict with references to the pieces of the boxplot
        data, # the list of arrays, one box drawn per array
        positions=x, # where on x-axis each box sits
        widths=0.55, # box width in x-axis units
        patch_artist=True, # required to color the box faces (otherwise boxes are just outlines)
        showfliers=False, # hide the default outlier dots, going to draw all points ourselves below
        medianprops=dict(color="black", linewidth=1.5),  # styling for the median line
        boxprops=dict(linewidth=1.2),  # styling for the box outline
        whiskerprops=dict(linewidth=1.0), # styling for the whisker lines
        capprops=dict(linewidth=1.0)# styling for the whisker end caps
    )
    

    # Step 7-- Color each box by genotype 
    for patch, g in zip(box["boxes"], GENOTYPE_ORDER):  # pair each box with its genotype name (same order)
        patch.set_facecolor(GENOTYPE_COLORS.get(g, "lightgray"))  # look up color, fall back to lightgray if missing
        patch.set_edgecolor("black")  # black border for contrast
        patch.set_alpha(0.75) # 75% opaque so scatter dots show through


    # Step 8-- Overlay individual ROI points (jittered) 
    for i, y in enumerate(data): # loop through a list giving you position index and the array of densities for that genotype
        if len(y) == 0:  # skip if this genotype has no data for this marker
            continue
        jitter = (np.random.rand(len(y)) - 0.5) * 0.20  # random horizontal wobble within a range --
                                                        # generate len(y) random nums between 0 and 1, - 0.5 and then * 0.20 to shrink the range
                                                        # now nums span -0.10 to 0.10, those are the "wobble values"
        ax.scatter( # draws the individual dots, one dot = one point (ROI); map each x-coord entry to y-coord entry 
            np.full(len(y), x[i]) + jitter,  # x coords: make an array of len(y) entries where every entry is value x[i] (the x-position) 
                                             # then adjust by the jitter ("wobble values")
            y,  # y coords: the density values themselves (the array of all of that genotype's ROI densities)
            color="black",
            s=18, # dot size (points^2)
            alpha=0.8, # slight transparency
            zorder=3, # dots sit ON boxes
        )

    # Step 9-- Overlay mean and SD error bars 
    # NOTE: if ecolor (color of error bars) conflicts with a genotype box color in GENOTYPE_COLORS, change ecolor below 
    
    means = [np.mean(y) if len(y) > 0 else np.nan for y in data]  # NaN for empty genotypes so matplotlib just skips 
    stds  = [np.std(y)  if len(y) > 0 else np.nan for y in data]  # one mean + SD density value per genotype
    ax.errorbar( 
        x, means,
        yerr=stds, # vertical error bar length is +-1 SD
        fmt="D",  # distinctive diamond marker for the mean
        color="black",
        markersize=8,
        markeredgecolor="white",   # white border so it pops against the box
        markeredgewidth=1.5,  
        ecolor="#1f77b4",  # color of the SD lines + caps (blue here)
        elinewidth=1.5,  # thickness of vertical SD lines
        capsize=4, # short horizontal caps at error bar ends
        linewidth=1.2,
        zorder=4, # draw ON scatter dots
    )

    # Step 10-- Build clean marker name for title 
    marker = col.replace("_cells_per_mm2", "").upper()  # Example: "cd8_cells_per_mm2" becomes "CD8"


    # Step 11-- Axis labels, ticks, + title 
    ax.set_xticks(x)  # put a tick at each genotype position 
    ax.set_xticklabels(GENOTYPE_ORDER, rotation=0)  # replace numbers with genotype names
    ax.set_xlabel("Genotype")
    ax.set_ylabel("Cells per mm^2")
    ax.set_title(f"{marker} cell density")

    if USE_LOG_SCALE: # optional log y-axis (set in config cell)
        ax.set_yscale("log")

    # Step 12-- Visual edits 
    ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)  # lightly dashed horizontal grid
    ax.set_axisbelow(True) # grid sits behind boxes/dots, not on top
    ax.spines["top"].set_visible(False)  # remove top frame line
    ax.spines["right"].set_visible(False) # remove right frame line
    plt.tight_layout()  # auto adjust margins so labels don't get cut off weirdly

    # Step 13-- Save the figure 
    try:
        filename = f"{marker}_density.png"    
        save_path = SAVE_DIR / filename # full output path using pathlib
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches="tight")  # trim whitespace as well
        print(f" --> saved: {filename}")
    except Exception as e: # if save fails 
        print(f"Could not save {col}: {e}")  # warn (give error message) but keep looping through remaining markers
    finally:
        plt.close(fig) # always close figure to free memory (this will still run even if the save failed)
print("\n All plots complete!")

cd8_cells_per_mm2 | test_run: n=4
 --> saved: CD8_density.png
cd4_cells_per_mm2 | test_run: n=4
 --> saved: CD4_density.png
macrophage_cells_per_mm2 | test_run: n=4
 --> saved: MACROPHAGE_density.png
nk_cells_per_mm2 | test_run: n=4
 --> saved: NK_density.png

 All plots complete!
